# LLMLingua Integration

Neural prompt compression using Microsoft's LLMLingua-2 for structured token reduction.

## How It Works

1. **Load the LLMLingua-2 model** (`microsoft/llmlingua-2-xlm-roberta-large-meetingbank`)
2. **Format the input** as a context + instruction + question bundle
3. **Compress to target token budget** while preserving answer-critical information
4. **Fallback to semantic pruning** if LLMLingua is unavailable or fails

## Trade-offs

| Pros | Cons |
|------|------|
| Neural compression preserves meaning | Requires GPU for optimal speed |
| Can achieve 65%+ compression | Model download required (~400MB) |
| Handles complex dependencies | Slower than regex/semantic methods |

## Requirements

- `llmlingua` package (`pip install llmlingua`)
- Falls back to semantic chunk pruning if unavailable

## Section A: Setup and Imports

In [ ]:
import importlib
import re
import numpy as np
from typing import List, Optional
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Check if LLMLingua is available
LLMLINGUA_AVAILABLE = False
try:
    importlib.import_module('llmlingua')
    LLMLINGUA_AVAILABLE = True
except ImportError:
    pass

print(f'LLMLingua available: {LLMLINGUA_AVAILABLE}')

## Section B: Helper Functions

In [ ]:
PROMPT_COMPRESSOR = None


def token_count(text: str) -> int:
    """Fast token count approximation (words * 1.3)."""
    if not text:
        return 0
    return int(len(text.split()) * 1.3)


def chunk_text(text: str, chunk_size_words: int = 180, overlap_words: int = 40) -> List[str]:
    """Split text into overlapping chunks."""
    words = text.split()
    if not words:
        return []

    chunks = []
    step = max(1, chunk_size_words - overlap_words)
    for i in range(0, len(words), step):
        chunk = words[i:i + chunk_size_words]
        if chunk:
            chunks.append(' '.join(chunk))
    return chunks

## Section C: Fallback Semantic Pruning

In [ ]:
def semantic_chunk_prune(text: str, question: str, top_k: int = 5) -> str:
    """Fallback: TF-IDF based semantic pruning."""
    chunks = chunk_text(text)
    if not chunks:
        return ''

    vect = TfidfVectorizer(ngram_range=(1, 2), min_df=1)
    vect.fit([question] + chunks)

    q_vec = vect.transform([question])
    c_vec = vect.transform(chunks)
    scores = cosine_similarity(q_vec, c_vec).flatten()

    idx = np.argsort(scores)[::-1][:min(top_k, len(chunks))]
    selected = [chunks[i] for i in idx]
    return '\n\n'.join(selected)

## Section D: LLMLingua Compression Method

In [ ]:
def llmlingua_prune(text: str, question: str, target_ratio: float = 0.35) -> str:
    """
    Compress context using LLMLingua-2 neural prompt compression.

    Args:
        text: The full document text
        question: The user's query
        target_ratio: Target compression ratio (0.35 = keep 35% of tokens)

    Returns:
        Compressed text preserving answer-critical information
    """
    global PROMPT_COMPRESSOR

    target_words = max(150, int(len(text.split()) * target_ratio))
    target_tokens = max(200, int(token_count(text) * target_ratio))

    if LLMLINGUA_AVAILABLE:
        try:
            if PROMPT_COMPRESSOR is None:
                from llmlingua import PromptCompressor
                PROMPT_COMPRESSOR = PromptCompressor(
                    model_name='microsoft/llmlingua-2-xlm-roberta-large-meetingbank',
                    use_llmlingua2=True,
                    device_map='cpu'
                )

            compression_input = f'Context:\n{text}\n\nQuestion: {question}'
            compressed = PROMPT_COMPRESSOR.compress_prompt(
                context=compression_input,
                instruction='Keep only information required to answer the question accurately.',
                question=question,
                target_token=target_tokens
            )

            if isinstance(compressed, dict):
                compressed_text = (
                    compressed.get('compressed_prompt')
                    or compressed.get('prompt')
                    or compressed.get('text')
                    or ''
                )
            else:
                compressed_text = str(compressed)

            if compressed_text.strip():
                return ' '.join(compressed_text.split()[:target_words])
        except Exception as e:
            print(f'LLMLingua error: {e}. Falling back to semantic pruning.')

    # Fallback to semantic pruning
    print('Using semantic pruning fallback.')
    reduced = semantic_chunk_prune(text, question, top_k=5)
    words = reduced.split()
    return ' '.join(words[:target_words])

## Section E: Demo and Testing

In [ ]:
sample_text = (
    "The company reported strong Q3 earnings. Revenue grew by 15% year over year. "
    "The expansion of the Frankfurt server farm cost $14.73 million. This was partially "
    "offset by a $2.1 million green energy tax rebate. Employee headcount remained stable. "
    "Management raised full-year guidance citing strong demand in European markets."
)

sample_question = "What is the projected infrastructure capital expenditure for Q3 2026?"

original_tokens = token_count(sample_text)
print(f"Original length: {len(sample_text.split())} words (~{original_tokens} tokens)")
print("\n" + "="*60)

result = llmlingua_prune(sample_text, sample_question, target_ratio=0.5)
compressed_tokens = token_count(result)
reduction = (1 - compressed_tokens / original_tokens) * 100 if original_tokens > 0 else 0

print(f"\nCompressed length: {len(result.split())} words (~{compressed_tokens} tokens)")
print(f"Compression: {reduction:.1f}%")
print(f"\nPruned output:\n{result}")

## Section F: Full Benchmark Interface

Upload a document, enter a question and your OpenRouter API key, then run a full comparison
across all four methods (Raw Baseline, Naive Regex, Semantic Chunk, LLMLingua).

### Setup: Install & Import Dependencies


In [ ]:
# Install missing packages (run once per session)
# !pip install -q openai pandas numpy plotly scikit-learn pypdf llmlingua

import importlib
import json
import os
import re
import textwrap
import time
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
from openai import OpenAI
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Optional PDF support
PDF_AVAILABLE = False
try:
    importlib.import_module('pypdf')
    PDF_AVAILABLE = True
except ImportError:
    pass

try:
    importlib.import_module('llmlingua')
    LLMLINGUA_AVAILABLE = True
except ImportError:
    LLMLINGUA_AVAILABLE = False

PROMPT_COMPRESSOR = None
print(f'pypdf: {PDF_AVAILABLE} | llmlingua: {LLMLINGUA_AVAILABLE}')

### Benchmark Functions


In [ ]:
# ----- Token & pricing helpers -----
PRICING_USD_PER_1K = {
    ('openrouter', 'meta-llama/llama-3.3-70b-instruct:free'): {'input': 0.0, 'output': 0.0},
    ('openrouter', 'google/gemma-4-31b-it:free'): {'input': 0.0, 'output': 0.0},
    ('openrouter', 'openrouter/free'): {'input': 0.0, 'output': 0.0},
}


def token_count(text: str) -> int:
    if not text:
        return 0
    return int(len(text.split()) * 1.3)


def compute_cost_usd(model: str, input_tokens: int, output_tokens: int) -> float:
    prices = PRICING_USD_PER_1K.get(('openrouter', model), {'input': 0.0, 'output': 0.0})
    return (input_tokens / 1000.0) * prices['input'] + (output_tokens / 1000.0) * prices['output']


# ----- OpenRouter client -----
def create_openrouter_client(api_key: str) -> OpenAI:
    key = api_key.strip() or os.getenv('OPENROUTER_API_KEY', '').strip()
    if not key:
        raise EnvironmentError('Paste your OpenRouter API key in the field below or set OPENROUTER_API_KEY env var.')
    return OpenAI(api_key=key, base_url='https://openrouter.ai/api/v1')


# ----- LLM caller -----
def call_llm_answer(client: OpenAI, model: str, context: str, question: str) -> Tuple[str, int, int, float]:
    prompt = (
        f"You are a precise analyst. Answer strictly from the provided context.\n\n"
        f"Context:\n{context}\n\nQuestion: {question}\n\n"
        f"If information is missing, say so clearly."
    )
    start = time.perf_counter()
    resp = client.chat.completions.create(
        model=model,
        messages=[
            {'role': 'system', 'content': 'You answer with concise, factual, context-grounded responses.'},
            {'role': 'user', 'content': prompt},
        ],
        temperature=0,
    )
    latency = time.perf_counter() - start
    answer = resp.choices[0].message.content if resp.choices else ''
    usage = getattr(resp, 'usage', None)
    in_tok = getattr(usage, 'prompt_tokens', token_count(prompt))
    out_tok = getattr(usage, 'completion_tokens', token_count(answer))
    return answer, in_tok, out_tok, latency


# ----- Document loading -----
def get_uploaded_text(upload_widget) -> Tuple[str, str]:
    value = upload_widget.value
    if isinstance(value, dict) and value:
        item = next(iter(value.values()))
    elif isinstance(value, (list, tuple)) and len(value) > 0:
        item = value[0]
    else:
        raise ValueError('Upload a file first.')

    name = item.get('name') or item.get('metadata', {}).get('name') or 'document.txt'
    content = item.get('content')
    if content is None:
        raise ValueError('Uploaded file content is missing.')

    if name.lower().endswith('.pdf'):
        if not PDF_AVAILABLE:
            raise ImportError('Install pypdf to parse PDF files: pip install pypdf')
        from io import BytesIO
        from pypdf import PdfReader
        reader = PdfReader(BytesIO(content))
        text = '\n'.join(page.extract_text() or '' for page in reader.pages)
    else:
        text = content.decode('utf-8', errors='ignore')

    return name, re.sub(r'\s+', ' ', text).strip()


# ----- Other pruning methods (needed for full comparison) -----
def _naive_regex_prune(text: str, question: str, max_sentences: int = 30) -> str:
    stop_words = {
        'the', 'is', 'at', 'which', 'what', 'on', 'and', 'a', 'an', 'in', 'of',
        'to', 'for', 'it', 'with', 'as', 'by', 'that', 'this', 'are', 'was',
        'be', 'or', 'from', 'we', 'not', 'do', 'if', 'no', 'so', 'up', 'all',
        'about', 'just', 'can', 'has', 'its', 'also', 'will', 'how', 'why',
    }
    sentences = re.split(r'(?<=[.!?])\s+', text)
    keywords = {w.lower() for w in re.findall(r'\b[a-zA-Z]{3,}\b', question) if w.lower() not in stop_words}
    if not keywords:
        return ' '.join(sentences[:max_sentences])
    scored = sorted([(sum(1 for kw in keywords if kw in s.lower()), s) for s in sentences], key=lambda x: -x[0])
    matched = [s for s, sc in scored if sc > 0]
    return ' '.join(matched[:max_sentences]) if matched else ' '.join(sentences[:max_sentences])


def _chunk_text(text: str, chunk_size: int = 180, overlap: int = 40) -> List[str]:
    words = text.split()
    chunks = []
    start = 0
    while start < len(words):
        end = min(start + chunk_size, len(words))
        chunk = words[start:end]
        if chunk:
            chunks.append(' '.join(chunk))
        if end == len(words):
            break
        start += chunk_size - overlap
    return chunks


def _semantic_chunk_prune(text: str, question: str, top_k: int = 8) -> str:
    chunks = _chunk_text(text)
    if not chunks:
        return ''
    vect = TfidfVectorizer(ngram_range=(1, 2), min_df=1)
    vect.fit([question] + chunks)
    q_vec = vect.transform([question])
    c_vec = vect.transform(chunks)
    scores = cosine_similarity(q_vec, c_vec).flatten()
    idx = np.argsort(scores)[::-1][:min(top_k, len(chunks))]
    return '\n\n'.join(chunks[i] for i in idx)


def _llmlingua_prune(text: str, question: str, target_ratio: float = 0.35) -> str:
    global PROMPT_COMPRESSOR
    target_words = max(150, int(len(text.split()) * target_ratio))
    target_tokens = max(200, int(token_count(text) * target_ratio))
    if LLMLINGUA_AVAILABLE:
        try:
            if PROMPT_COMPRESSOR is None:
                from llmlingua import PromptCompressor
                PROMPT_COMPRESSOR = PromptCompressor(
                    model_name='microsoft/llmlingua-2-xlm-roberta-large-meetingbank',
                    use_llmlingua2=True, device_map='cpu',
                )
            compressed = PROMPT_COMPRESSOR.compress_prompt(
                context=f'Context:\n{text}\n\nQuestion: {question}',
                instruction='Keep only information required to answer the question accurately.',
                question=question,
                target_token=target_tokens,
            )
            if isinstance(compressed, dict):
                ct = compressed.get('compressed_prompt') or compressed.get('prompt') or compressed.get('text') or ''
            else:
                ct = str(compressed)
            if ct.strip():
                return ' '.join(ct.split()[:target_words])
        except Exception:
            pass
    reduced = _semantic_chunk_prune(text, question, top_k=5)
    return ' '.join(reduced.split()[:target_words])


# ----- Build all contexts -----
def build_method_contexts(raw_text: str, question: str) -> Dict[str, str]:
    return {
        'Raw Baseline': raw_text,
        'Naive': _naive_regex_prune(raw_text, question),
        'Semantic': _semantic_chunk_prune(raw_text, question),
        'LLMLingua': _llmlingua_prune(raw_text, question),
    }


# ----- LLM-as-a-Judge -----
def llm_judge_scores(client: OpenAI, judge_model: str, question: str, gold_answer: str, candidate: str) -> Dict[str, float]:
    resp = client.chat.completions.create(
        model=judge_model,
        messages=[
            {'role': 'system', 'content': 'You are a strict evaluator. Return only JSON.'},
            {'role': 'user', 'content': (
                f'You are an impartial evaluator.\n\nQuestion: {question}\n\n'
                f'Gold Standard Answer:\n{gold_answer}\n\nCandidate Answer:\n{candidate}\n\n'
                f'Score candidate from 1-10 for each metric:\n'
                f'- Contextual Recall\n- Contextual Precision\n- Answer Correctness\n- Faithfulness\n\n'
                f'Return only valid JSON with keys: contextual_recall, contextual_precision, answer_correctness, faithfulness.'
            )},
        ],
        temperature=0,
    )
    text = (resp.choices[0].message.content or '').strip() if resp.choices else ''
    match = re.search(r'\{.*?\}', text, flags=re.DOTALL)
    if not match:
        raise ValueError(f'Judge output is not JSON: {text}')
    data = json.loads(match.group(0))
    return {
        'Contextual Recall': float(data['contextual_recall']),
        'Contextual Precision': float(data['contextual_precision']),
        'Answer Correctness': float(data['answer_correctness']),
        'Faithfulness': float(data['faithfulness']),
    }

### Benchmark Runner


In [ ]:
METHOD_ORDER = ['Raw Baseline', 'Naive', 'Semantic', 'LLMLingua']

status_out = None
table_out = None
judge_out = None


def run_benchmark(file_uploader, question_input, model_dropdown, judge_dropdown, api_key_input):
    import IPython.display as ip_display

    try:
        filename, raw_text = get_uploaded_text(file_uploader)
        question = question_input.value.strip()
        if not question:
            raise ValueError('Please enter a question.')
        answer_model = model_dropdown.value
        judge_model = judge_dropdown.value
        inline_key = api_key_input.value.strip()

        client = create_openrouter_client(inline_key)

        print(f'File: {filename}')
        print(f'Question: {question}')
        print(f'Answer model: {answer_model}')
        print(f'Judge model: {judge_model}')
        print('Building contexts...')

        contexts = build_method_contexts(raw_text, question)
        original_tokens = token_count(raw_text)
        records = []
        answers = {}

        for method in METHOD_ORDER:
            ctx = contexts[method]
            print(f'  Running: {method} ...', end=' ', flush=True)
            comp_tokens = token_count(ctx)
            answer, in_tok, out_tok, latency = call_llm_answer(client, answer_model, ctx, question)
            cost = compute_cost_usd(answer_model, in_tok, out_tok)
            reduction = max(0.0, (1 - (comp_tokens / max(1, original_tokens))) * 100.0)
            if method == 'Raw Baseline':
                reduction = 0.0
            answers[method] = answer
            records.append({
                'Method': method,
                'Original Tokens': original_tokens,
                'Compressed Tokens': comp_tokens,
                'Compression Ratio %': round(reduction, 2),
                'Input Tokens': in_tok,
                'Output Tokens': out_tok,
                'Cost USD': round(cost, 6),
                'Latency (s)': round(latency, 3),
            })
            print(f'{latency:.1f}s')

        metrics_df = pd.DataFrame(records)
        gold_answer = answers['Raw Baseline']

        print('\nJudging...')
        judge_rows = []
        for method in METHOD_ORDER:
            print(f'  Judging: {method}')
            score = llm_judge_scores(client, judge_model, question, gold_answer, answers[method])
            judge_rows.append({'Method': method, **score})
        judge_df = pd.DataFrame(judge_rows)[
            ['Method', 'Contextual Recall', 'Contextual Precision', 'Answer Correctness', 'Faithfulness']
        ]

        ip_display.display(ip_display.Markdown('### Comparison Table'))
        try:
            ip_display.display(metrics_df.style.background_gradient(cmap='YlGnBu',
                subset=['Compressed Tokens', 'Compression Ratio %', 'Latency (s)', 'Cost USD']))
        except Exception:
            ip_display.display(metrics_df)

        ip_display.display(ip_display.Markdown('### LLM-as-a-Judge Scores (1-10)'))
        try:
            ip_display.display(judge_df.style.background_gradient(cmap='YlOrBr',
                subset=['Contextual Recall', 'Contextual Precision', 'Answer Correctness', 'Faithfulness']))
        except Exception:
            ip_display.display(judge_df)

        ip_display.display(ip_display.Markdown('### Gold Standard Answer (Raw Baseline)'))
        print(textwrap.shorten(gold_answer, width=3000, placeholder=' ...[truncated]'))

        ip_display.display(ip_display.Markdown('### Method Answers'))
        for method in METHOD_ORDER:
            if method == 'Raw Baseline':
                continue
            ip_display.display(ip_display.Markdown(f'#### {method}'))
            print(textwrap.shorten(answers.get(method, ''), width=3000, placeholder=' ...[truncated]'))
            print('')

        print('Benchmark complete.')

    except Exception as e:
        print(f'Error: {type(e).__name__}: {e}')


print('Runner ready.')

### Launch UI


In [ ]:
import ipywidgets as widgets
from IPython.display import display

MODEL_OPTIONS = ['meta-llama/llama-3.3-70b-instruct:free', 'google/gemma-4-31b-it:free', 'openrouter/free']

file_uploader = widgets.FileUpload(accept='.txt,.md,.pdf', multiple=False, description='Upload File')

question_input = widgets.Textarea(
    placeholder='Ask a specific question about the uploaded document...',
    description='Question:',
    layout=widgets.Layout(width='100%', height='120px')
)

model_input = widgets.Dropdown(
    options=MODEL_OPTIONS,
    value='meta-llama/llama-3.3-70b-instruct:free',
    description='Answer Model:'
)

judge_input = widgets.Dropdown(
    options=MODEL_OPTIONS,
    value='meta-llama/llama-3.3-70b-instruct:free',
    description='Judge Model:'
)

api_key_input = widgets.Password(
    value='',
    description='OpenRouter API Key:',
    placeholder='Paste your OpenRouter API key here'
)

run_button = widgets.Button(description='Run Full Benchmark', button_style='success', icon='play')

output = widgets.Output()


def on_run(_):
    with output:
        output.clear_output()
        run_benchmark(file_uploader, question_input, model_input, judge_input, api_key_input)


run_button.on_click(on_run)

display(
    widgets.VBox([
        file_uploader,
        question_input,
        widgets.HBox([model_input, judge_input]),
        api_key_input,
        run_button,
        output,
    ])
)